# 05. 추론 · 제출

|  |  |
|---|---|
| **입력** | `runs/<실험>/weights/best.pt`, 테스트 이미지 폴더, `data/pill_yolo/category_map.json` |
| **출력** | `outputs/submissions/*.csv`, `outputs/predictions/*/` |

---

## 제출 형식

```
annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score
1, 1, 1900, 156, 247, 211, 456, 0.91
```

- 한 행 = 검출 객체 하나
- `annotation_id` 는 전체에서 1부터 이어지는 고유 번호
- `bbox` 는 COCO 형식 `[좌상단x, 좌상단y, 너비, 높이]` 절대 픽셀
- 검출이 없는 이미지는 행이 없어도 됩니다

## ★ 조용히 망하는 세 지점

**① 전처리 누락 — 이번 버전에서 고쳤습니다**
02 는 학습 이미지에 **화이트밸런스 + CLAHE** 를 걸어 저장했습니다.
추론에서 이걸 안 걸면 모델이 처음 보는 색분포가 들어갑니다.
mAP 가 이유 없이 낮게 나오는 가장 흔한 원인입니다.
아래 셀은 `category_map.json` 의 `preprocess` 기록을 읽어 **자동으로 동일 전처리**를 겁니다.

**② `category_id` 역매핑**
YOLO 는 0..N-1 인덱스를 출력합니다. 이걸 원래 `category_id` (1900, 16548...)로
되돌리지 않으면 **예측은 정확한데 점수만 0** 이 나옵니다.

**③ `image_id` 매핑**
파일명과 `image_id` 가 다르면 전부 어긋납니다.

## 해상도도 맞추세요
`IMGSZ` 는 **03 의 학습 해상도와 같아야** 합니다.
학습 960, 추론 640 이면 각인이 안 읽혀 오분류가 급증합니다.

In [ ]:
# ═══════════════ 설정 ═══════════════
DATA_ROOT = r"D:/PillData"
#1. 팀 구글드라이브에 있는 PillData의 압축을 푼다. 
#2. 새로운 파일을 생성한다.
#3. 그 파일에 압축을 푼 파일을 넣고, 새로운 파일의 경로주소를 적는다.
# ex)D드라이브 안에 있는 X라는 이름의 파일에 PillData파일을 넣었다. 그럼 D:/X 로 설정

EXP_NAME  = "exp_offline"                         # ★ 03 의 EXP_NAME 과 동일하게

WEIGHTS   = f"{DATA_ROOT}/runs/{EXP_NAME}/weights/best.pt"
TEST_IMG  = r"C:/Users/User/Desktop/Notannotationpng"   # 테스트 이미지 폴더
TEST_ANN  = f"{DATA_ROOT}/pilldata/test_annotations.json"  # 없으면 "" 로 두세요
CMAP_JSON = f"{DATA_ROOT}/data/pill_yolo/category_map.json"

IMGSZ   = 960        # ★ 03 의 IMGSZ 와 반드시 동일하게
CONF    = 0.001      # ★ mAP 용이므로 낮게
IOU_NMS = 0.7
MAX_DET = 100
TOPK    = 4          # 이미지당 상위 N개 (0=전부). 대회 제약이 '최대 4개'라면 4
TTA     = False      # True 면 느리지만 보통 +점수

# ★ 학습과 동일한 전처리를 적용할지. category_map.json 기록을 따르는 것이 기본
APPLY_PREPROCESS = True

OUT_ROOT  = f"{DATA_ROOT}/outputs"
SUB_CSV   = f"{OUT_ROOT}/submissions/{EXP_NAME}_top{TOPK}.csv"
VIS_DIR   = f"{OUT_ROOT}/predictions/{EXP_NAME}_test"
PRE_DIR   = f"{OUT_ROOT}/_preprocessed_test"      # 전처리된 임시 이미지
VIS_CONF  = 0.25     # 시각화용 (제출과 별개)
VIS_N     = 8

SEED = 42

import os
for d in (os.path.dirname(SUB_CSV), VIS_DIR, PRE_DIR, f"{DATA_ROOT}/experiments"):
    os.makedirs(d, exist_ok=True)
print(f"가중치   : {WEIGHTS}  존재 {os.path.exists(WEIGHTS)}")
print(f"테스트   : {TEST_IMG}")
print(f"제출 파일: {SUB_CSV}")

In [ ]:
"""공통 유틸 — 이 셀을 먼저 실행하세요."""
import os, json, glob, csv, random, shutil
from collections import Counter, defaultdict

import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont

random.seed(SEED); np.random.seed(SEED)


# ---------------------------------------------------------------- 한글 경로 IO
def imread_unicode(path):
    try:
        return cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    except Exception:
        return None


def imwrite_unicode(path, img):
    ext = os.path.splitext(str(path))[1] or ".png"
    ok, buf = cv2.imencode(ext, img)
    if not ok:
        return False
    buf.tofile(str(path)); return True


# ---------------------------------------------------------------- 한글 폰트
def find_korean_font():
    cands = ["C:/Windows/Fonts/malgun.ttf",
             "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
             "/System/Library/Fonts/AppleSDGothicNeo.ttc"]
    for pat in ("/usr/share/fonts/**/*CJK*.ttc", "/usr/share/fonts/**/*Nanum*.ttf",
                "/usr/share/fonts/**/*Gothic*.ttf"):
        cands += sorted(glob.glob(pat, recursive=True))
    for p in cands:
        if os.path.exists(p):
            try:
                ImageFont.truetype(p, 20); return p
            except Exception:
                pass
    return None

FONT_PATH = find_korean_font()
print("한글 폰트:", FONT_PATH or "⚠️ 미발견")

_PALETTE = [(255,89,94),(56,176,0),(25,130,196),(255,202,58),(138,80,220),
            (0,187,249),(241,91,181),(155,200,60),(255,140,0),(0,200,170),
            (200,60,120),(120,160,255)]

def class_color(cid):
    return _PALETTE[int(cid) % len(_PALETTE)]


def draw_detections(img_bgr, dets, id2name=None, font_scale=1.0,
                    box_thickness=3, show_conf=True):
    """★ 라벨 형식: "약이름 0.91" """
    img = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)
    H, W = img_bgr.shape[:2]
    size = max(16, int(min(W, H) * 0.028 * font_scale))
    font = ImageFont.truetype(FONT_PATH, size) if FONT_PATH else ImageFont.load_default()
    for d in dets:
        cid = d["category_id"]
        x, y, w, h = [float(v) for v in d["bbox"]]
        color = class_color(cid)
        name = str(id2name.get(cid, cid)) if id2name else str(cid)
        label = f"{name} {d['score']:.2f}" if (show_conf and d.get("score") is not None) else name
        draw.rectangle([x, y, x + w, y + h], outline=color, width=box_thickness)
        tb = draw.textbbox((0, 0), label, font=font)
        tw, th = tb[2] - tb[0], tb[3] - tb[1]
        pad = max(3, size // 6)
        ly = y - th - pad * 2
        if ly < 0:
            ly = y + pad
        lx = min(max(0, x), W - tw - pad * 2)
        draw.rectangle([lx, ly, lx + tw + pad*2, ly + th + pad*2], fill=color)
        lum = 0.299*color[0] + 0.587*color[1] + 0.114*color[2]
        draw.text((lx+pad, ly+pad), label, font=font,
                  fill=(0,0,0) if lum > 150 else (255,255,255))
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def save_per_image(records, img_dir, out_dir, id2name=None, conf_thr=0.0,
                   limit=0, suffix="", verbose=True):
    """★ 이미지 한 장당 결과 파일 한 개씩."""
    os.makedirs(out_dir, exist_ok=True)
    n = 0
    for fn, dets in sorted(records.items()):
        p = os.path.join(img_dir, fn)
        if not os.path.exists(p):
            p = os.path.join(img_dir, os.path.basename(fn))
            if not os.path.exists(p):
                continue
        img = imread_unicode(p)
        if img is None:
            continue
        keep = [d for d in dets if d.get("score") is None or d["score"] >= conf_thr]
        vis = draw_detections(img, keep, id2name)
        stem, ext = os.path.splitext(os.path.basename(fn))
        imwrite_unicode(os.path.join(out_dir, f"{stem}{suffix}{ext or '.png'}"), vis)
        n += 1
        if limit and n >= limit:
            break
    if verbose:
        print(f"{n}장 저장 → {out_dir}  (이미지 1장 = 파일 1개)")
    return n


def show_image(path, max_width=900):
    from IPython.display import display
    im = Image.open(path)
    if im.width > max_width:
        im = im.resize((max_width, int(im.height * max_width / im.width)))
    display(im)


# ══════════ ★ 학습과 동일한 전처리 (02 의 Preprocess 와 같은 식) ══════════
CMAP = json.load(open(CMAP_JSON, encoding="utf-8"))
PRE_CFG = CMAP.get("preprocess") or {}
USE_WB    = bool(PRE_CFG.get("white_balance", False))
USE_CLAHE = bool(PRE_CFG.get("clahe_L", False))
CLAHE_CLIP = float(PRE_CFG.get("clahe_clip", 2.0))


def shades_of_gray(img_bgr, p=6):
    f = img_bgr.astype(np.float32)
    norm = np.power(np.power(f, p).mean(axis=(0, 1)), 1.0 / p)
    norm = np.maximum(norm, 1e-6)
    return np.clip(f * (norm.mean() / norm), 0, 255).astype(np.uint8)


def clahe_on_L(img_bgr, clip=2.0, grid=8):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)


def preprocess(img_bgr):
    """★ 02 의 학습 인코딩과 동일. 순서도 같아야 합니다."""
    out = img_bgr
    if USE_WB:
        out = shades_of_gray(out)
    if USE_CLAHE:
        out = clahe_on_L(out, CLAHE_CLIP)
    return out


# ---------------------------------------------------------------- 매핑
IDX2CAT = {int(k): int(v) for k, v in CMAP["idx2cat"].items()}
IDXNAME = {int(k): v for k, v in CMAP["names"].items()}
# ★ names 는 YOLO 인덱스 키 → category_id 로 다시 매핑
ID2NAME = {IDX2CAT[i]: n for i, n in IDXNAME.items()}

print(f"\n클래스 {len(IDX2CAT)}종  YOLO 0~{max(IDX2CAT)} → category_id")
print(f"학습 시 전처리 기록: WB={USE_WB}, CLAHE={USE_CLAHE} (clip={CLAHE_CLIP})")
print(f"추론 전처리 적용   : {APPLY_PREPROCESS and (USE_WB or USE_CLAHE)}")
if APPLY_PREPROCESS and not (USE_WB or USE_CLAHE):
    print("  (학습에 전처리가 없었으므로 추론에도 적용하지 않습니다 — 정상)")
print(f"02 증강 기록: {CMAP.get('augment')}")

---
## 1. image_id 매핑 + 전처리

### image_id 우선순위
1. `test_annotations.json` 의 `images` 목록 ← 가장 정확
2. 파일명(확장자 제외)이 숫자면 그걸 id 로
3. 정렬 순서 1부터 ← 최후 수단, **틀릴 위험 큼**

### ★ 전처리
학습 이미지와 같은 상태로 만들기 위해 전처리한 사본을 `_preprocessed_test/` 에
만들고 **그 사본으로 예측**합니다. 원본은 건드리지 않습니다.
좌표는 리사이즈 없이 그대로이므로 **박스를 되돌릴 필요가 없습니다.**

In [ ]:
paths = sorted(sum([glob.glob(f"{TEST_IMG}/{e}")
                    for e in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")], []))
print(f"테스트 이미지 {len(paths):,}장")
if not paths:
    raise SystemExit(f"{TEST_IMG} 에 이미지가 없습니다. 경로를 확인하세요.")

# ---------- image_id ----------
id_map = {}
if TEST_ANN and os.path.exists(TEST_ANN):
    with open(TEST_ANN, encoding="utf-8") as f:
        by_name = {os.path.basename(i["file_name"]): i["id"]
                   for i in json.load(f).get("images", [])}
    miss = 0
    for p in paths:
        b = os.path.basename(p)
        if b in by_name:
            id_map[p] = by_name[b]
        else:
            miss += 1
    print(f"image_id 출처: test_annotations.json (매칭 {len(id_map)} / 미매칭 {miss})")
    if miss:
        print("⚠️  일부가 json 에 없습니다. 파일명 규칙을 확인하세요.")

if not id_map:
    stems = [os.path.splitext(os.path.basename(p))[0] for p in paths]
    if all(s.isdigit() for s in stems):
        id_map = {p: int(s) for p, s in zip(paths, stems)}
        print("image_id 출처: 파일명 숫자")
    else:
        id_map = {p: i + 1 for i, p in enumerate(paths)}
        print("⚠️  image_id 출처: 정렬 순서 — 정확하지 않을 수 있습니다!")

# ---------- ★ 전처리 사본 ----------
do_pre = APPLY_PREPROCESS and (USE_WB or USE_CLAHE)
pred_paths, path_of_pred = [], {}

if do_pre:
    for p in glob.glob(f"{PRE_DIR}/*"):
        os.remove(p)
    n_fail = 0
    for k, p in enumerate(paths, 1):
        img = imread_unicode(p)
        if img is None:
            n_fail += 1; continue
        dst = os.path.join(PRE_DIR, os.path.basename(p))
        imwrite_unicode(dst, preprocess(img))
        pred_paths.append(dst); path_of_pred[dst] = p
        if k % 200 == 0:
            print(f"  전처리 {k}/{len(paths)}")
    print(f"전처리 완료 {len(pred_paths)}장 → {PRE_DIR}"
          + (f" (읽기 실패 {n_fail}장)" if n_fail else ""))
    PRED_IMG_DIR = PRE_DIR
else:
    pred_paths = list(paths)
    path_of_pred = {p: p for p in paths}
    PRED_IMG_DIR = TEST_IMG
    print("전처리 미적용 (학습에도 없었거나 APPLY_PREPROCESS=False)")

# ---------- 전처리 전/후 비교 1장 ----------
if do_pre and pred_paths:
    src = imread_unicode(path_of_pred[pred_paths[0]])
    dst = imread_unicode(pred_paths[0])
    if src is not None and dst is not None:
        gap = np.full((src.shape[0], 8, 3), 255, np.uint8)
        cmp_path = f"{OUT_ROOT}/figures/{EXP_NAME}_test_preprocess.png"
        os.makedirs(os.path.dirname(cmp_path), exist_ok=True)
        imwrite_unicode(cmp_path, np.hstack([src, gap, dst]))
        print("\n좌 = 원본 / 우 = 전처리 (학습 이미지와 같은 상태여야 합니다)")
        show_image(cmp_path, max_width=1000)

---
## 2. 추론 및 제출 파일 생성

`TOPK` 는 이미지당 남길 검출 수입니다.
원본 데이터가 **이미지당 알약 3~4개**이므로 `TOPK=4` 가 자연스럽지만,
mAP 지표에서는 낮은 confidence 예측도 recall 에 기여하므로
`TOPK=0`(전부)이 더 높게 나올 수도 있습니다. **둘 다 제출해 비교하세요.**
파일명에 `top{TOPK}` 가 들어가므로 덮어쓰지 않습니다.

In [ ]:
from ultralytics import YOLO
from datetime import datetime

model = YOLO(WEIGHTS)
rows, ann_id = [], 1
per_img, confs = Counter(), []
vis_records = {}

t0 = datetime.now()
for i in range(0, len(pred_paths), 8):
    chunk = pred_paths[i:i+8]
    results = model.predict(chunk, conf=CONF, iou=IOU_NMS, imgsz=IMGSZ,
                            max_det=MAX_DET, augment=TTA, verbose=False)
    for p, r in zip(chunk, results):
        orig = path_of_pred[p]
        iid = id_map[orig]
        if r.boxes is None or len(r.boxes) == 0:
            per_img[0] += 1; continue
        xyxy = r.boxes.xyxy.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        cls  = r.boxes.cls.cpu().numpy().astype(int)
        order = conf.argsort()[::-1]
        if TOPK:
            order = order[:TOPK]

        dets_vis = []
        for j in order:
            x1, y1, x2, y2 = xyxy[j]
            w, h = float(x2-x1), float(y2-y1)
            if w <= 0 or h <= 0:
                continue
            cid = IDX2CAT[int(cls[j])]          # ★ YOLO 인덱스 → 원본 category_id
            rows.append([ann_id, iid, cid, round(float(x1), 2), round(float(y1), 2),
                         round(w, 2), round(h, 2), round(float(conf[j]), 5)])
            ann_id += 1; confs.append(float(conf[j]))
            dets_vis.append({"category_id": cid, "bbox": [float(x1), float(y1), w, h],
                             "score": float(conf[j])})
        per_img[len(order)] += 1
        if len(vis_records) < VIS_N:
            vis_records[os.path.basename(p)] = dets_vis
    if (i // 8) % 20 == 0:
        print(f"  {min(i+8, len(pred_paths))}/{len(pred_paths)}")

with open(SUB_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["annotation_id", "image_id", "category_id",
                "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"])
    w.writerows(rows)

print(f"\n소요 시간: {datetime.now() - t0}")
print(f"검출 {len(rows):,}개 / 이미지 {len(pred_paths):,}장 "
      f"(평균 {len(rows)/max(1,len(pred_paths)):.1f}개)")
print(f"저장: {SUB_CSV}")

In [ ]:
print("■ 이미지당 검출 개수")
for k in sorted(per_img):
    print(f"    {k:>3}개 : {per_img[k]:>5,}장")
if per_img.get(0):
    print(f"    ⚠️  검출 0개 이미지가 {per_img[0]}장 → CONF 를 더 낮추거나 학습을 더 하세요")
exp_range = sum(v for k, v in per_img.items() if 3 <= k <= 4)
print(f"    알약 3~4개로 검출된 이미지 {exp_range:,}장 "
      f"({exp_range/max(1,len(pred_paths)):.0%}) ← 원본 분포와 비슷해야 자연스럽습니다")

if confs:
    cs = np.array(confs)
    print("\n■ confidence 분포")
    for t in (0.001, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9):
        n = int((cs >= t).sum())
        print(f"    ≥{t:<6} {n:>7,}개 ({n/len(cs)*100:5.1f}%)")
    print(f"    중앙값 {np.median(cs):.4f}")
    if np.median(cs) < 0.05:
        print("\n    ⚠️  중앙값이 매우 낮습니다. 대부분이 저확신 예측입니다.")
        print("       mAP 에는 도움이 될 수 있으나, 학습이 충분한지 점검하세요.")
        print("       전처리 누락이 원인인 경우가 많습니다 — 위 비교 이미지를 확인하세요.")

# ★ category_id 가 원본 값인지 확인 (0~N-1 이면 역매핑 실패)
if rows:
    cids = {r[2] for r in rows}
    print(f"\n■ category_id 범위 {min(cids)} ~ {max(cids)}",
          "✅ 원본 값" if min(cids) > 100 else "❌ YOLO 인덱스입니다! 역매핑 확인")
    unknown = cids - set(ID2NAME)
    print(f"  매핑에 없는 category_id {len(unknown)}개",
          "✅" if not unknown else f"❌ {sorted(unknown)[:5]}")

---
## 3. 예측 시각화 — **이미지 1장 = 파일 1개**

여러 장을 격자에 몰아넣으면 알약과 라벨이 작아져 확인이 어렵습니다.
**한 장씩 따로** 저장하면 가시성이 좋고 보고서에 개별로 넣기도 편합니다.

라벨 형식은 **`약이름 confidence`** — 약 이름 뒤에 한 칸 띄우고 확신도가 붙습니다.

> 전처리된 이미지 위에 그립니다. 모델이 실제로 본 화면이 그것이기 때문입니다.

In [ ]:
save_per_image(vis_records, PRED_IMG_DIR, VIS_DIR,
               id2name=ID2NAME, conf_thr=VIS_CONF, suffix="_pred")
print(f"\n(conf ≥ {VIS_CONF} 만 표시. 제출 파일과는 별개입니다)")

for p in sorted(glob.glob(f"{VIS_DIR}/*"))[:5]:
    print("\n" + os.path.basename(p))
    show_image(p)

---
## 4. 제출 기록

캐글에 올린 뒤 **점수를 여기에 채워 넣으세요.** 보고서 그래프 재료가 됩니다.

In [ ]:
SUB_LOG = f"{DATA_ROOT}/experiments/submissions.jsonl"

rec = {"file": SUB_CSV, "exp": EXP_NAME,
       "timestamp": datetime.now().isoformat(timespec="seconds"),
       "weights": WEIGHTS, "conf": CONF, "iou": IOU_NMS, "imgsz": IMGSZ,
       "topk": TOPK, "tta": TTA,
       "preprocess_applied": bool(do_pre),
       "offline_aug": CMAP.get("augment"),
       "n_detections": len(rows), "n_images": len(pred_paths),
       "kaggle_score": None}          # ← 제출 후 직접 채워 넣으세요

with open(SUB_LOG, "a", encoding="utf-8") as f:
    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"기록: {SUB_LOG}")
print("★ 캐글 점수가 나오면 kaggle_score 를 채워 넣으세요")

In [ ]:
# 제출 이력 요약 (kaggle_score 를 채운 뒤 실행)
if os.path.exists(SUB_LOG):
    recs = []
    for line in open(SUB_LOG, encoding="utf-8"):
        try:
            d = json.loads(line)
        except json.JSONDecodeError:
            continue
        if "exp" in d:                    # 형식이 다른 옛 레코드는 건너뜀
            recs.append(d)

    print(f"{'실험':<16}{'topk':>5}{'전처리':>7}{'conf':>8}{'검출수':>9}{'캐글점수':>10}")
    print("-" * 56)
    for r in recs:
        s = r.get("kaggle_score")
        print(f"{r['exp'][:15]:<16}{r['topk']:>5}"
              f"{('O' if r.get('preprocess_applied') else 'X'):>7}"
              f"{r['conf']:>8}{r['n_detections']:>9,}"
              f"{(f'{s:.4f}' if s is not None else '-'):>10}")

    scored = [r for r in recs if r.get("kaggle_score") is not None]
    if len(scored) >= 2:
        import matplotlib.pyplot as plt
        from matplotlib import font_manager
        if FONT_PATH:
            font_manager.fontManager.addfont(FONT_PATH)
            plt.rcParams["font.family"] = font_manager.FontProperties(fname=FONT_PATH).get_name()
        plt.rcParams["axes.unicode_minus"] = False
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(range(len(scored)), [r["kaggle_score"] for r in scored],
                "o-", color="#3a86ff")
        ax.set_xticks(range(len(scored)))
        ax.set_xticklabels([r["exp"] for r in scored], rotation=25)
        ax.set_ylabel("Kaggle score"); ax.set_title("제출별 점수 추이")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        os.makedirs(f"{OUT_ROOT}/figures", exist_ok=True)
        plt.savefig(f"{OUT_ROOT}/figures/kaggle_progress.png", bbox_inches="tight")
        plt.show()
    else:
        print("\n(kaggle_score 를 2개 이상 채우면 추이 그래프가 나옵니다)")

---
## 5. 제출 전 체크리스트

- [ ] **전처리를 학습과 동일하게 걸었는가** (위 비교 이미지 확인) ← 가장 흔한 실수
- [ ] `IMGSZ` 가 03 의 학습 해상도와 같은가
- [ ] `category_id` 가 원래 값(1900, 16548...)인가? 0~N-1 이면 **잘못된 것**
- [ ] `image_id` 가 test json 기준으로 매핑됐는가
- [ ] `bbox` 가 `[x, y, w, h]` 형식인가 (`x1,y1,x2,y2` 아님)
- [ ] `annotation_id` 가 1부터 고유한가
- [ ] 시각화 이미지에서 박스가 알약에 정확히 씌워지는가

아래 셀이 위 항목 중 자동 확인 가능한 것을 검사합니다.

## TOPK 비교 실험

| TOPK | 의미 | 캐글 점수 |
|---|---|---|
| 0 | 전부 제출 (mAP 유리?) | |
| 10 | 상위 10개 | |
| 4 | 상위 4개 (대회 제약과 일치) | |

설정 셀에서 `TOPK` 만 바꾸고 이 노트북을 다시 실행하세요.

In [ ]:
import pandas as pd

df = pd.read_csv(SUB_CSV)
print(df.head())
print(f"\n행 수                 {len(df):,}")
print(f"category_id 범위      {df.category_id.min()} ~ {df.category_id.max()}",
      "✅" if df.category_id.min() > 100 else "❌ YOLO 인덱스입니다")
print(f"image_id 범위         {df.image_id.min()} ~ {df.image_id.max()}")
print(f"annotation_id 고유    {df.annotation_id.is_unique}",
      "✅" if df.annotation_id.is_unique else "❌")
print(f"annotation_id 1부터   {df.annotation_id.min() == 1}",
      "✅" if df.annotation_id.min() == 1 else "❌")
print(f"bbox 양수             {bool((df.bbox_w > 0).all() and (df.bbox_h > 0).all())}",
      "✅" if (df.bbox_w > 0).all() and (df.bbox_h > 0).all() else "❌")
print(f"score 범위            {df.score.min():.5f} ~ {df.score.max():.5f}")
print(f"이미지당 평균 검출     {len(df)/df.image_id.nunique():.2f}개")
print(f"전처리 적용           {do_pre}",
      "✅" if do_pre == (USE_WB or USE_CLAHE) else "⚠️ 학습 설정과 다릅니다")

if TOPK:
    over = df.groupby("image_id").size().max()
    print(f"이미지당 최대 검출     {over} / TOPK {TOPK}",
          "✅" if over <= TOPK else "❌")